# 09 – Evaluation: Importancia de Variables

**Proyecto:** Análisis y predicción del subempleo por insuficiencia de horas en el Perú – EPEN 2024  
**Target:** `target_subempleo_horas` (1 = subempleado por horas · 0 = no subempleado)  
**Objetivo:** Analizar qué variables tienen mayor impacto predictivo usando dos enfoques:

1. **MDI (Mean Decrease Impurity)** – desde el Random Forest GridSearchCV (`random_forest_optimized.pkl`), ya versionado en `tree_feature_importance.csv`.
2. **Importancia por Permutación** – sobre el modelo ganador (`logistic_regression_balanced.pkl`), que es el que maximiza F1 y Recall de la clase 1.
3. **Coeficientes de la Regresión Logística** – interpretación directa del modelo ganador.

> **Prerequisito:** Ejecuta `06_feature_selection/selected_variables.ipynb`, `07_modelling/01_baseline_model.ipynb` y `07_modelling/02_decision_tree_or_random_forest.ipynb`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

# ── Rutas ──────────────────────────────────────────────────────────────────────
SEL_DIR     = Path('../data/selected')
MODEL_DIR   = Path('../models')
RESULTS_DIR = Path('../data/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET      = 'target_subempleo_horas'
CLASS_NAMES = ['No subempleado por horas', 'Subempleado por horas']

# ── Carga de datos reales (sin fallback sintético) ─────────────────────────────
required = {
    'X_train': SEL_DIR / 'X_train_selected.csv',
    'X_test':  SEL_DIR / 'X_test_selected.csv',
    'y_train': SEL_DIR / 'y_train_selected.csv',
    'y_test':  SEL_DIR / 'y_test_selected.csv',
}
for name, path in required.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Archivo requerido no encontrado: {path}\n"
            "Ejecuta primero: 06_feature_selection/selected_variables.ipynb"
        )

X_train = pd.read_csv(required['X_train'])
X_test  = pd.read_csv(required['X_test'])

def load_target(path, name):
    df = pd.read_csv(path)
    if TARGET in df.columns:
        return df[TARGET].reset_index(drop=True)
    if df.shape[1] == 1:
        return df.iloc[:, 0].reset_index(drop=True)
    raise ValueError(f"No se encontró '{TARGET}' en {name}")

y_train = load_target(required['y_train'], 'y_train')
y_test  = load_target(required['y_test'],  'y_test')

# ── Carga del modelo ganador (LR balanceada) para permutación y coeficientes ───
lr_path = MODEL_DIR / 'logistic_regression_balanced.pkl'
if not lr_path.exists():
    raise FileNotFoundError(
        f"Modelo no encontrado: {lr_path}\n"
        "Ejecuta primero: 07_modelling/01_baseline_model.ipynb"
    )
model_lr = joblib.load(lr_path)

# ── Carga del RF optimizado para MDI ──────────────────────────────────────────
rf_path = MODEL_DIR / 'random_forest_optimized.pkl'
if not rf_path.exists():
    raise FileNotFoundError(
        f"Modelo no encontrado: {rf_path}\n"
        "Ejecuta primero: 07_modelling/02_decision_tree_or_random_forest.ipynb"
    )
model_rf = joblib.load(rf_path)

feature_names = X_train.columns.tolist()
print(f'Datos cargados: {X_test.shape[0]} observaciones, {len(feature_names)} variables.')
print(f'Modelo ganador (LR): {lr_path.name}')
print(f'Modelo RF para MDI : {rf_path.name}')


## 1. Importancia por Impureza Media (MDI) – Random Forest GridSearchCV

Usando el Random Forest optimizado (no el modelo ganador). MDI refleja qué variables el RF usa más para reducir impureza. Se complementa con permutación e interpretación del modelo ganador (LR).


In [ ]:
# MDI desde el RF optimizado (ya versionado en tree_feature_importance.csv)
importances_mdi = pd.Series(model_rf.feature_importances_, index=feature_names) \
                    .sort_values(ascending=True)

n_show = min(30, len(feature_names))
fig, ax = plt.subplots(figsize=(10, max(5, n_show * 0.35)))
importances_mdi.tail(n_show).plot(kind='barh', color='steelblue', edgecolor='black', ax=ax)
ax.set_title('Top variables – MDI (Random Forest GridSearchCV)')
ax.set_xlabel('Importancia relativa (reducción de impureza)')
plt.tight_layout()
plt.show()

print('\nTop 10 variables por MDI:')
print(importances_mdi.sort_values(ascending=False).head(10).to_string())


## 2. Coeficientes del Modelo Ganador – Regresión Logística Balanceada

Los coeficientes de la LR representan la contribución de cada variable al log-odds de ser subempleado por horas. Valor positivo → mayor probabilidad de clase 1; negativo → menor.


In [ ]:
# Extraer coeficientes del pipeline LR (último paso named 'clf' o similar)
if hasattr(model_lr, 'named_steps'):
    lr_step = model_lr.named_steps.get('clf', list(model_lr.named_steps.values())[-1])
else:
    lr_step = model_lr

coefs = pd.Series(lr_step.coef_[0], index=feature_names).sort_values()

n_show = min(30, len(feature_names))
fig, axes = plt.subplots(1, 2, figsize=(16, max(5, n_show * 0.35)))

# Coeficientes negativos (reducen prob. subempleo)
coefs_neg = coefs[coefs < 0].tail(15)
coefs_neg.plot(kind='barh', color='steelblue', edgecolor='black', ax=axes[0])
axes[0].set_title('Top coeficientes NEGATIVOS\n(reducen P(subempleo))')
axes[0].set_xlabel('Coeficiente')

# Coeficientes positivos (aumentan prob. subempleo)
coefs_pos = coefs[coefs > 0].tail(15)
coefs_pos.plot(kind='barh', color='tomato', edgecolor='black', ax=axes[1])
axes[1].set_title('Top coeficientes POSITIVOS\n(aumentan P(subempleo))')
axes[1].set_xlabel('Coeficiente')

plt.suptitle('Coeficientes – Logistic Regression (class_weight="balanced")',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

print('\nTop 10 positivos (mayor riesgo de subempleo):')
print(coefs.sort_values(ascending=False).head(10).to_string())
print('\nTop 10 negativos (menor riesgo de subempleo):')
print(coefs.sort_values().head(10).to_string())


In [ ]:
## 3. Importancia por Permutación – Modelo Ganador (Logistic Regression Balanceada)

Mide la caída en ROC-AUC al permutar aleatoriamente cada variable. Funciona para cualquier modelo.


In [ ]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    model_lr, X_test, y_test,
    n_repeats=10, random_state=42, scoring='roc_auc', n_jobs=-1
)

perm_imp = pd.DataFrame({
    'Variable': feature_names,
    'Mean':     perm_result.importances_mean,
    'Std':      perm_result.importances_std,
}).sort_values('Mean', ascending=True)

n_show = min(30, len(feature_names))
fig, ax = plt.subplots(figsize=(10, max(5, n_show * 0.35)))
subset = perm_imp.tail(n_show)
ax.barh(subset['Variable'], subset['Mean'],
        xerr=subset['Std'], color='darkorange', edgecolor='black', capsize=3)
ax.set_title('Importancia por Permutación – LR balanceada (ROC-AUC)')
ax.set_xlabel('Reducción promedio en ROC-AUC al permutar la variable')
plt.tight_layout()
plt.show()

print('\nTop 10 variables por permutación:')
print(perm_imp.sort_values('Mean', ascending=False).head(10)[['Variable', 'Mean', 'Std']].to_string(index=False))


## 4. SHAP Values (opcional – requiere `pip install shap`)

Proporciona explicaciones a nivel de observación. Úsalo si necesitas interpretar predicciones individuales.


In [ ]:
try:
    import shap
    # Para LR: usar LinearExplainer
    explainer = shap.LinearExplainer(model_lr, X_train)
    shap_values = explainer.shap_values(X_test)

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
    plt.title('SHAP – Importancia media (clase: Subempleado por horas)')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('SHAP no está instalado. Ejecuta: pip install shap')
    print('Los métodos MDI, coeficientes y permutación son suficientes.')
except Exception as e:
    print(f'SHAP no disponible para este modelo: {e}')
    print('Usa los coeficientes LR y la importancia por permutación como alternativa.')


In [ ]:
# Guardar tabla de importancias MDI (RF) y de permutación (LR)
importances_mdi.sort_values(ascending=False) \
    .to_frame('MDI_RF_Importance') \
    .to_csv(RESULTS_DIR / 'feature_importance_final.csv')

perm_imp.sort_values('Mean', ascending=False) \
    .rename(columns={'Mean': 'Perm_LR_Mean', 'Std': 'Perm_LR_Std'}) \
    .to_csv(RESULTS_DIR / 'feature_importance_permutation_lr.csv', index=False)

print(f'Guardado: {RESULTS_DIR / "feature_importance_final.csv"}')
print(f'Guardado: {RESULTS_DIR / "feature_importance_permutation_lr.csv"}')
